# Tl;DR

Download the datasets (from the root of the repository), either manually from Zenodo (DOI [10.5281/zenodo.23244510](https://doi.org/10.5281/zenodo.23244510)) \
or by running:

```bash
python src/load_data.py --dim all --chi all   # everything: D = 2 to 16, plus 23 and 32, both chi
python src/load_data.py --dim 2 --chi 43      # a single dimension, a single chi (faster)
python src/load_data.py --dim 2 3 4 --chi 30  # several dimensions
```

Build a new randomised `(N, D)` point cloud from the downloaded ones:

```bash
python src/new_data.py --dim 2 --npoints 1024 --chi 43 --seed 42
```

or from Python:

```python
from new_data import new_data
X = new_data(N=1024, D=2, chi=43, seed=42, data_dir="data")   # (N, D) float32 array
```

**The `chi` parameter** (`30` or `43`) selects the dataset. It controls the spectral properties: `chi = 30` (χ = 0.30) gives `S(k) <= 1e-10` on a smaller exclusion region, `chi = 43` (χ = 0.43) gives `S(k) <= 1e-4` on a larger one. A `chi` cloud can only be built from the `chi` datasets: download the matching ones first.

`new_data.py` performs, in this order:

1. **Projection**: if `D` is not stored (`16 < D <= 32`, `D` not 23 or 32), `D` coordinates are drawn at random from the `D = 23` (`D <= 23`) or `D = 32` dataset.
2. **Tiling**: if `N >= 2^18` (about 260k), the largest stored cloud (`N = 2^17`) is tiled, see [TILING.md](TILING.md).
3. **Randomisation** (with the given seed): random shift `x_new = (x - s) mod 1`, random flips `x_new[:, d] = 1 - x_new[:, d]`, and random axis swaps.

**Supported:** any `N` power of 2 with `2^10 <= N <= 2^24`, any `2 <= D <= 32`, `chi` in `{30, 43}`.

> There is no hard limit on tiling, but we recommend not going far beyond `N = 2^24` (16M points): stealthiness degrades proportionally to `N_t / N`.

# I. Load the datasets from Zenodo

In [ ]:
# Run from the root of the repository. Files go to data/chi{chi}/D{D}/N{i}_D{D}.h5
!python src/load_data.py --dim 2 --chi 43   # or --dim 2 3 4 / --dim all, and --chi 30 / --chi all

# II. Create a new dataset

A new dataset is generated from the downloaded ones by exploiting the symmetries of the distribution:

* random shifts
* random flips
* random axis swaps

When needed (see TL;DR), projection and tiling are applied first, taking a point cloud from shape `(N, D)` to `(N × 2^i, D − j)`.

In [ ]:
!python src/new_data.py --dim 2 --npoints 1024 --chi 43 --seed 42

# This is equivalent to:
# import numpy as np
# from new_data import new_data
# X = new_data(N=1024, D=2, chi=43, seed=42)
# np.save("new_data/N1024_D2_chi43_seed42.npy", X)

# III. Analysis

This requires the `blue-sampler` package (`pip install blue-sampler`).

In [ ]:
import sys
sys.path.insert(0, "src")

import blue_sampler as blue
from new_data import new_data

X = new_data(N=1024, D=2, chi=43, seed=42)
blue.plot(X)
blue.plot_structure_factor(X)